In [1]:
import os
import time
import hashlib
import pandas as pd
from selenium.webdriver.common.by import By
import undetected_chromedriver as uc

In [2]:
# Leer el token desde el archivo externo
ruta_token = "auth.tk"
try:
    with open(ruta_token, "r", encoding="utf-8") as archivo:
        mi_auth_token = archivo.read().strip()
except FileNotFoundError:
    print(f"ERROR: No se encontró el archivo '{ruta_token}'.")
    print("Crea el archivo en la misma carpeta del script, pega tu token dentro y vuelve a ejecutar.")
    driver.quit()
    exit()

In [ ]:
def seudonimizar_usuario(username):
    """Aplica un hash SHA-256 al nombre de usuario para cumplir con la Ley 21.719."""
    if not username: return None
    return hashlib.sha256(username.encode('utf-8')).hexdigest()

def extraer_comentarios_x(url, periodo, driver, limite=300):
    driver.get(url)
    print(f"Cargando {periodo}...")
    time.sleep(6) # Pausa inicial para carga de DOM
    
    comentarios_extraidos = []
    ids_vistos = set()
    intentos_sin_nuevos = 0 
    botones_cliqueados = set() # <--- Memoria para no repetir clics en el mismo botón
    
    while len(comentarios_extraidos) < limite:
        
        # Salvaguarda de navegación
        url_base = url.split('?')[0]
        if not driver.current_url.startswith(url_base):
            print("El navegador se desvió de la página principal. Regresando...")
            driver.get(url)
            time.sleep(5)
            
        cantidad_anterior = len(comentarios_extraidos)
        articulos = driver.find_elements(By.XPATH, '//article[@data-testid="tweet"]')
        
        for articulo in articulos:
            try:
                # 1. Obtener enlace único del tweet
                link_elem = articulo.find_elements(By.XPATH, './/a[contains(@href, "/status/")]')
                tweet_id = link_elem[0].get_attribute('href') if link_elem else None
                
                if not tweet_id or tweet_id in ids_vistos:
                    continue
                
                # 2. Extraer texto del tweet
                texto_elem = articulo.find_elements(By.XPATH, './/div[@data-testid="tweetText"]')
                texto = texto_elem[0].text if texto_elem else None
                
                # 3. Detectar si contiene multimedia
                media_elem = articulo.find_elements(By.XPATH, './/div[@data-testid="tweetPhoto"] | .//div[@data-testid="videoPlayer"]')
                tiene_media = len(media_elem) > 0
                
                if not texto:
                    if tiene_media:
                        texto = "[Contenido multimedia sin texto]"
                    else:
                        continue 
                
                # 4. Extraer usuario y fecha
                user_elem = articulo.find_elements(By.XPATH, './/div[@dir="ltr"]/span')
                usuario_raw = user_elem[0].text if user_elem else "usuario_desconocido"
                
                time_elem = articulo.find_elements(By.XPATH, './/time')
                fecha_hora = time_elem[0].get_attribute('datetime') if time_elem else None
                
                ids_vistos.add(tweet_id)
                
                comentarios_extraidos.append({
                    "id_texto": len(ids_vistos),
                    "fecha_hora": fecha_hora,
                    "id_usuario_hash": seudonimizar_usuario(usuario_raw),
                    "texto": texto,
                    "consulta_origen": url,
                    "periodo": periodo
                })
                
                if len(comentarios_extraidos) >= limite:
                    break
                    
            except Exception:
                continue 
                
        if len(comentarios_extraidos) >= limite:
            break
            
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3) 
        
        # Control de scroll y botones de expansión / spam
        if len(comentarios_extraidos) == cantidad_anterior:
            intentos_sin_nuevos += 1
            
            try:
                botones = driver.find_elements(By.XPATH, "//*[contains(text(), 'Mostrar') or contains(text(), 'spam') or contains(text(), 'Show')]")
                for btn in botones:
                    try:
                        # Identificamos el botón por su contenido único
                        btn_id = btn.get_attribute('outerHTML')
                        if btn.is_displayed() and btn_id not in botones_cliqueados:
                            driver.execute_script("arguments[0].click();", btn)
                            botones_cliqueados.add(btn_id) # Registramos que ya fue presionado
                            print("Botón de 'Mostrar' o 'spam' presionado con éxito...")
                            time.sleep(3)
                            break
                    except Exception:
                        continue
            except Exception:
                pass
                
            # Si tras 5 intentos el scroll ya no trae comentarios nuevos, salimos limpiamente
            if intentos_sin_nuevos >= 5: 
                print(f"Fondo de la página alcanzado. Obtenidos: {len(comentarios_extraidos)} de {limite}.")
                break
        else:
            intentos_sin_nuevos = 0 
            print(f"Extrayendo... {len(comentarios_extraidos)} / {limite}")
            
    return comentarios_extraidos[:limite]
# ==========================================
# 1. Configurar el WebDriver Indetectable
# ==========================================
ruta_perfil_local = os.path.abspath(os.path.join(os.getcwd(), "perfil_selenium_uc"))
if not os.path.exists(ruta_perfil_local):
    os.makedirs(ruta_perfil_local)

opciones = uc.ChromeOptions()
opciones.add_argument(f"--user-data-dir={ruta_perfil_local}") 
opciones.add_argument("--disable-dev-shm-usage")

opciones.add_argument("--lang=en-US")
prefs = {"intl.accept_languages": "en,en_US"}
opciones.add_experimental_option("prefs", prefs)

# Iniciar el navegador. Si sigues con el error de versión, usa: uc.Chrome(options=opciones, version_main=155)
driver = uc.Chrome(options=opciones, version_main=155)

# ==========================================
# 2. Login por Inyección de Cookie
# ==========================================
print("Inyectando sesión de X...")
driver.get("https://x.com/robots.txt")
time.sleep(2)


driver.add_cookie({
    'name': 'auth_token',
    'value': mi_auth_token,
    'domain': '.x.com'
})

# Refrescar la página principal
driver.get("https://x.com/")
time.sleep(5)
print("Sesión inyectada con éxito. Iniciando extracción...")

# ==========================================
# 3. Recolección de Datos
# ==========================================
links = [
    {"url": "https://x.com/NintendoAmerica/status/2064361089668620543?s=20", "periodo": "Antes (Expectativa / Anuncio)"},
    {"url": "https://x.com/NintendoAmerica/status/2097333400054927619?s=20", "periodo": "Después (Tráiler Gameplay)"}
]

datos_totales = []
for link in links:
    datos = extraer_comentarios_x(link["url"], link["periodo"], driver, limite=500)
    datos_totales.extend(datos)

driver.quit()

# ==========================================
# 4. Estructurar en Pandas y exportar
# ==========================================
df_comentarios = pd.DataFrame(datos_totales)

# Prevenir error si la muestra solicitada es mayor a los datos obtenidos
muestra_size = min(50, len(df_comentarios))
df_muestra_ruido = df_comentarios.sample(n=muestra_size, random_state=42) if muestra_size > 0 else pd.DataFrame()

df_comentarios.to_csv("corpus_ocarina_remake.csv", sep=';', index=False, encoding="utf-8")
if not df_muestra_ruido.empty:
    df_muestra_ruido.to_csv("muestra_ruido_manual.csv", sep=';', index=False, encoding="utf-8")

print("\nExtracción completada. Primeras 5 filas:")
print(df_comentarios.head())

Inyectando sesión de X...
Sesión inyectada con éxito. Iniciando extracción...
Cargando Antes (Expectativa / Anuncio)...
Extrayendo... 5 / 300
Extrayendo... 19 / 300
Extrayendo... 41 / 300
Extrayendo... 56 / 300
Extrayendo... 74 / 300
Extrayendo... 91 / 300
Extrayendo... 112 / 300
Extrayendo... 130 / 300
Extrayendo... 157 / 300
Extrayendo... 186 / 300
Extrayendo... 219 / 300
Extrayendo... 222 / 300
Botón de 'Mostrar' o 'spam' presionado con éxito...
Extrayendo... 231 / 300
Extrayendo... 232 / 300
Botón de 'Mostrar' o 'spam' presionado con éxito...
El navegador se desvió de la página principal. Regresando...
Fondo de la página alcanzado. Obtenidos: 232 de 300.
Cargando Después (Tráiler Gameplay)...
Extrayendo... 5 / 300
Extrayendo... 22 / 300
Extrayendo... 44 / 300
Extrayendo... 55 / 300
Extrayendo... 69 / 300
Extrayendo... 87 / 300
Extrayendo... 110 / 300
Extrayendo... 134 / 300
Extrayendo... 159 / 300
Extrayendo... 179 / 300
Extrayendo... 197 / 300
Extrayendo... 198 / 300
Botón de 'Mos

In [7]:
import pandas as pd

df_comentarios = pd.read_csv("corpus_ocarina_remake.csv", sep=';', encoding="utf-8")
df_comentarios.head()

,id_texto,fecha_hora,id_usuario_hash,texto,consulta_origen,periodo
0,1,2026-06-09T14:56:52.000Z,400b585eb60519835945bb0fa92adfd7977f7369241350...,The Legend of Zelda: Ocarina of Time will be r...,https://x.com/NintendoAmerica/status/206436108...,Antes (Expectativa / Anuncio)
1,2,2026-06-09T14:57:32.000Z,961d77778edf61e276b5a7952e551462685566d82021ac...,"""Hey can we see Ocarina of Time gameplay?""\n\n...",https://x.com/NintendoAmerica/status/206436108...,Antes (Expectativa / Anuncio)
2,3,2026-06-09T15:01:03.000Z,31ed9acf9453843e09587a7e6ed5b05b3d7e48921da4b9...,We're getting him in HD we are not ready for this,https://x.com/NintendoAmerica/status/206436108...,Antes (Expectativa / Anuncio)
3,4,2026-06-09T16:19:01.000Z,90a0d8e9c0416f10d8f02429c3af3c3f49a5f8a98278fc...,,https://x.com/NintendoAmerica/status/206436108...,Antes (Expectativa / Anuncio)
4,5,2026-06-09T16:17:01.000Z,4b6c621736bc9eaac24e6c4c82cd858fd60bebe4f12668...,My honest reaction to this information.,https://x.com/NintendoAmerica/status/206436108...,Antes (Expectativa / Anuncio)
